# Overview of the ML Pipeline

```

University Dataset
        ↓
Clean Data
        ↓
Generate Training Dataset
        ↓
Train ML Model
        ↓
Student Input
        ↓
Model Predict Match Score
        ↓
Rank Universities
```



In [58]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import random

# 1. Load Data

In [59]:
sheet_id = "1c-zyqbxZ1lTnZWQvGm2c77zHXB9XeDEzM4uFEhefIXM"

url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

df = pd.read_csv(url)

df.head()

,University,Location,GradeA,GradeB,GradeC,GradeE,Quota,EntranceExam,Majors
0,សាកលវិទ្យាល័យវិទ្យាស្ថានអាហ្កា - AGA Universit...,Phnom Penh,100,85,0,0,130,Yes,Marketing|Accounting|Management|Commerce|Econo...
1,សាកលវិទ្យាល័យ ពាណិជ្ជសាស្ត្រ អេស៊ីលីដា - ACLED...,Phnom Penh,100,80,50,0,115,Yes,Banking and Finance|International Business|Ris...
2,សាកលវិទ្យាល័យ ប៊ែលធី អន្តរជាតិ - BELTEI Intern...,Phnom Penh,100,75,0,0,90,No,Business Administration|Banking and Finance|Ec...
3,សាកលវិទ្យាល័យបូណាម៉ារី - Bonamari University,Phnom Penh,100,80,0,0,30,No,Khmer Literature (Social Science| Khmer Litera...
4,សាកលវិទ្យាល័យមេគង្គកម្ពុជា - Cambodia Mekong U...,Phnom Penh,100,90,0,0,125,Yes,Hotel and Tourism Management|Human Resource Ma...


# 2. Clean Dataset

In [60]:
col = ['Quota', 'Majors']

df[col].head()

,Quota,Majors
0,130,Marketing|Accounting|Management|Commerce|Econo...
1,115,Banking and Finance|International Business|Ris...
2,90,Business Administration|Banking and Finance|Ec...
3,30,Khmer Literature (Social Science| Khmer Litera...
4,125,Hotel and Tourism Management|Human Resource Ma...


In [61]:
df[col] = df[col].replace(0, np.nan)

df[col].head()

,Quota,Majors
0,130.0,Marketing|Accounting|Management|Commerce|Econo...
1,115.0,Banking and Finance|International Business|Ris...
2,90.0,Business Administration|Banking and Finance|Ec...
3,30.0,Khmer Literature (Social Science| Khmer Litera...
4,125.0,Hotel and Tourism Management|Human Resource Ma...


In [62]:
df.isnull().sum()

University       0
Location         0
GradeA           0
GradeB           0
GradeC           0
GradeE           0
Quota           10
EntranceExam     0
Majors           9
dtype: int64

In [63]:
df = df.dropna()

df.isnull().sum()

University      0
Location        0
GradeA          0
GradeB          0
GradeC          0
GradeE          0
Quota           0
EntranceExam    0
Majors          0
dtype: int64

In [64]:
#Convert numeric columns
numeric_cols = ["GradeA","GradeB","GradeC","GradeE","Quota"]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

#Replace missing numeric values
df[numeric_cols] = df[numeric_cols].fillna(0)

#Remove universities where all scholarships are 0
df = df[
    (df[["GradeA","GradeB","GradeC","GradeE"]].sum(axis=1) > 0)
]

#Final Clean Dataset Check
df.reset_index(drop=True, inplace=True)

df.head()

,University,Location,GradeA,GradeB,GradeC,GradeE,Quota,EntranceExam,Majors
0,សាកលវិទ្យាល័យវិទ្យាស្ថានអាហ្កា - AGA Universit...,Phnom Penh,100,85,0,0,130.0,Yes,Marketing|Accounting|Management|Commerce|Econo...
1,សាកលវិទ្យាល័យ ពាណិជ្ជសាស្ត្រ អេស៊ីលីដា - ACLED...,Phnom Penh,100,80,50,0,115.0,Yes,Banking and Finance|International Business|Ris...
2,សាកលវិទ្យាល័យ ប៊ែលធី អន្តរជាតិ - BELTEI Intern...,Phnom Penh,100,75,0,0,90.0,No,Business Administration|Banking and Finance|Ec...
3,សាកលវិទ្យាល័យបូណាម៉ារី - Bonamari University,Phnom Penh,100,80,0,0,30.0,No,Khmer Literature (Social Science| Khmer Litera...
4,សាកលវិទ្យាល័យមេគង្គកម្ពុជា - Cambodia Mekong U...,Phnom Penh,100,90,0,0,125.0,Yes,Hotel and Tourism Management|Human Resource Ma...


In [66]:
# Convert Entrance Exam
exam_map = {"No":0, "Yes":1}

df["EntranceExam"] = df["EntranceExam"].map(exam_map).fillna(0)

# Covert Grade
grade_map = {"A":5, "B":4, "C":3, "D":2, "E":1}

# 3. Generate Machine Learning Training Data

In [67]:
training_rows = []

majors_all = set()

for majors in df["Majors"]:
    majors_all.update([m.strip() for m in majors.split("|")])

majors_all = list(majors_all)

provinces = df["Location"].unique()

In [68]:
# Normalize Quota
df["quota_norm"] = df["Quota"] / df["Quota"].max()

In [69]:
for _, uni in df.iterrows():

    for grade in ["A","B","C","E"]:

        scholarship = uni[f"Grade{grade}"]

        if scholarship == 0:
            continue

        for _ in range(15):

            student_major = random.choice(majors_all)
            student_province = random.choice(provinces)

            major_list = [m.strip() for m in uni["Majors"].split("|")]

            major_match = 1 if student_major in major_list else 0
            province_match = 1 if student_province == uni["Location"] else 0

            student_grade_score = grade_map[grade]

            grade_diff = student_grade_score - 3

            scholarship_norm = scholarship / 100

            quota_norm = uni["quota_norm"]

            entrance_exam = uni["EntranceExam"]

            # smarter label rule
            match = 0

            if scholarship >= 50 and major_match == 1:
                match = 1

            training_rows.append({
                "student_grade": student_grade_score,
                "grade_diff": grade_diff,
                "scholarship_norm": scholarship_norm,
                "quota_norm": quota_norm,
                "entrance_exam": entrance_exam,
                "major_match": major_match,
                "province_match": province_match,
                "match": match
            })

In [44]:
# for _, uni in df.iterrows():
#
#     for grade in ["A","B","C","E"]:
#
#         scholarship = uni[f"Grade{grade}"]
#
#         if scholarship == 0:
#             continue
#
#         for _ in range(10):
#
#             student_major = random.choice(majors_all)
#             student_province = random.choice(provinces)
#
#             major_list = [m.strip() for m in uni["Majors"].split("|")]
#
#             major_match = 1 if student_major in major_list else 0
#             province_match = 1 if student_province == uni["Location"] else 0
#
#             match = 1 if (major_match == 1 and scholarship > 0) else 0
#
#             training_rows.append({
#                 "student_grade": grade_map[grade],
#                 "scholarship_percent": scholarship,
#                 "quota": uni["Quota"],
#                 "entrance_exam": uni["EntranceExam"],
#                 "major_match": major_match,
#                 "province_match": province_match,
#                 "match": match
#             })

In [70]:
training_df = pd.DataFrame(training_rows)

training_df.head()

,student_grade,grade_diff,scholarship_norm,quota_norm,entrance_exam,major_match,province_match,match
0,5,2,1.0,1.0,0.0,0,0,0
1,5,2,1.0,1.0,0.0,0,0,0
2,5,2,1.0,1.0,0.0,0,0,0
3,5,2,1.0,1.0,0.0,0,0,0
4,5,2,1.0,1.0,0.0,0,0,0


# 4. Train Machine Learning Model

In [71]:
X = training_df.drop("match", axis=1)
y = training_df["match"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=42
)

model.fit(X_train, y_train)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(1824, 7) (1824,)
(456, 7) (456,)


In [47]:
# X = training_df.drop("match", axis=1)
# y = training_df["match"]
#
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42
# )
#
# model = RandomForestClassifier()
#
# model.fit(X_train, y_train)
#
# print(X_train.shape, y_train.shape)
# print(X_test.shape, y_test.shape)

(1216, 6) (1216,)
(304, 6) (304,)


In [72]:
y_pred = model.predict(X_test)

print("Predicting is completed...")

Predicting is completed...


In [73]:
accuracy = accuracy_score(y_test, y_pred)

print(f"Among {len(y_test):d} university, our ML model classified {accuracy*100:.2f}% correctly")

Among 456 university, our ML model classified 100.00% correctly


# Recommendation Function Using ML

In [96]:
def recommend_universities(student_grade, major, province):

    results = []

    grade_score = grade_map[student_grade]

    province = province.lower()

    for _, uni in df.iterrows():

        if uni["Location"].lower() != province:
            continue

        major_list = [m.strip().lower() for m in uni["Majors"].split("|")]

        major_match = 1 if major.lower() in major_list else 0

        if major_match == 0:
            continue

        scholarship = uni[f"Grade{student_grade}"]

        if scholarship == 0:
            continue

        features = pd.DataFrame([{
            "student_grade": grade_score,
            "grade_diff": grade_score - 3,
            "scholarship_norm": scholarship / 100,
            "quota_norm": uni["quota_norm"],
            "entrance_exam": uni["EntranceExam"],
            "major_match": 1,
            "province_match": 1
        }])

        score = model.predict_proba(features)[0][1]

        results.append({
            "University": uni["University"],
            "Location": uni["Location"],
            "Scholarship": scholarship,
            "EntranceExam": "Yes" if uni["EntranceExam"] == 1 else "No",
            "Quota": uni["Quota"],
            "Score": round(score*100,2)
        })

    # Handle empty results
    if len(results) == 0:
        return pd.DataFrame({
            "Message": ["No universities found for this combination"]
        })

    results_df = pd.DataFrame(results)

    results_df = results_df.sort_values(by="Score", ascending=False)

    return results_df.head(5).reset_index(drop=True)

In [76]:
# def recommend_universities(student_grade, major, province):
#
#     results = []
#
#     grade_score = grade_map[student_grade]
#
#     province = province.lower()
#
#     for _, uni in df.iterrows():
#
#         # FILTER BY PROVINCE FIRST
#         if uni["Location"].lower() != province:
#             continue
#
#         major_list = [m.strip().lower() for m in uni["Majors"].split("|")]
#
#         major_match = 1 if major.lower() in major_list else 0
#
#         if major_match == 0:
#             continue
#
#         scholarship = uni[f"Grade{student_grade}"]
#
#         if scholarship == 0:
#             continue
#
#         features = pd.DataFrame([{
#             "student_grade": grade_score,
#             "scholarship_percent": scholarship,
#             "quota": uni["Quota"],
#             "entrance_exam": uni["EntranceExam"],
#             "major_match": 1,
#             "province_match": 1
#         }])
#
#         score = model.predict_proba(features)[0][1]
#
#         results.append({
#             "University": uni["University"],
#             "Location": uni["Location"],
#             "Scholarship": scholarship,
#             "Score": round(score*100,2)
#         })
#
#     results_df = pd.DataFrame(results)
#
#     results_df = results_df.sort_values(by="Score", ascending=False)
#
#     return results_df.head(5).reset_index(drop=True)

In [98]:
def recommend_universities(student_grade, major, province):

    results = []

    grade_score = grade_map[student_grade]

    province = province.lower()

    for _, uni in df.iterrows():

        # Filter province
        if uni["Location"].lower() != province:
            continue

        major_list = [m.strip().lower() for m in uni["Majors"].split("|")]

        # Check major
        if major.lower() not in major_list:
            continue

        scholarship = uni[f"Grade{student_grade}"]

        if scholarship == 0:
            continue

        features = pd.DataFrame([{
            "student_grade": grade_score,
            "grade_diff": grade_score - 3,
            "scholarship_norm": scholarship / 100,
            "quota_norm": uni["quota_norm"],
            "entrance_exam": uni["EntranceExam"],
            "major_match": 1,
            "province_match": 1
        }])

        score = model.predict_proba(features)[0][1]

        results.append({
            "university": uni["University"],
            "location": uni["Location"],
            "scholarship": scholarship,
            "quota": uni["Quota"],
            "score": round(score * 100, 2)
        })

    # If no results → return empty list
    if len(results) == 0:
        return []

    # Sort results
    results = sorted(results, key=lambda x: x["score"], reverse=True)

    return results[:5]

In [100]:
# Test the recommendation function

recommend_universities(
    student_grade="A",
    major="English",
    province="Battambang"
)

[{'university': 'សាកលវិទ្យាល័យអន្តរជាតិ ឌូវី - Dewey International University (DIU)',
  'location': 'Battambang',
  'scholarship': 100,
  'quota': 105.0,
  'score': np.float64(89.05)},
 {'university': 'សាកលវិទ្យាល័យគ្រប់គ្រង និងសេដ្ឋកិច្ច (បាត់ដំបង) - University of Management and Economics (UME)',
  'location': 'Battambang',
  'scholarship': 100,
  'quota': 112.0,
  'score': np.float64(89.05)},
 {'university': 'ពុទ្ធិកសាកលវិទ្យាល័យព្រះសីហនុរាជ (បាត់ដំបង) - Preah Sihanouk Raja Buddhist University (Battambang)',
  'location': 'Battambang',
  'scholarship': 90,
  'quota': 72.0,
  'score': np.float64(88.11)}]